# FinExpert v4: teacher-written targets (explanations + analysis-based suggestions)

A stronger **teacher** (Qwen3-14B, 4-bit) writes training targets for the v4 student:

| Set | Prompts | What the teacher writes | Automatic checks |
|---|---|---|---|
| Concept explanations | 234 (train terms only) | The whole explanation, in a fixed structure | Structure, length, arithmetic in the example, no advice |
| Analysis-based suggestions | 240 | Only the suggestions; the facts are computed, not generated | No verdict, no price prediction, only input figures, analysis, disclaimer, 2-4 bullets |

Failures are retried twice with sampling; accepted items are then scored by an LLM judge (rubric in `docs/v4_judge.md`);
items with a score below 4 on any criterion are dropped. A **blind sheet of 30 items** is written for you to score by hand.

**Colab:** Runtime → T4 GPU, set `COLAB_USE_DRIVE = True`, Run all, keep the tab open.
**Kaggle:** GPU T4 x2, Internet on, Save & Run All. First try `LIMIT = 10` (about 10 min) to check everything works.

## 1. Config

In [ ]:
RUN_NAME        = "v4-teacher"
REPO_URL        = "https://github.com/mani1234765/finexpert.git"
REPO_REF        = "main"
TEACHER_MODEL   = "unsloth/Qwen3-14B-unsloth-bnb-4bit"   # Apache 2.0; fits a 15 GB T4 in 4-bit
NO_THINK        = True       # Qwen3-14B reasons in <think> by default; an empty think block switches that off
LIMIT           = None       # e.g. 10 for a quick end-to-end test
GEN_BATCH_SIZE  = 8
MAX_NEW_TOKENS  = {"explain": 400, "assess": 200, "judge": 120}
COLAB_USE_DRIVE = True       # Colab only: copy results to Google Drive as they are produced

## 2. Platform and install

In [ ]:
import os, sys, subprocess, json, time
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
IN_KAGGLE = os.path.exists("/kaggle/working")
IN_COLAB = "google.colab" in sys.modules
WORK_DIR = "/kaggle/working" if IN_KAGGLE else ("/content" if IN_COLAB else os.getcwd())
REPO_DIR = "/tmp/finexpert-repo"
RUN_DIR = os.path.join(WORK_DIR, "runs", RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)

def sh(*cmd, cwd=None):
    r = subprocess.run(list(cmd), capture_output=True, text=True, cwd=cwd)
    if r.returncode != 0:
        print(r.stdout[-2000:], r.stderr[-2000:]); raise RuntimeError(" ".join(cmd))
    return r.stdout.strip()

import torch
assert torch.cuda.is_available(), "No GPU: Colab Runtime > T4 GPU; Kaggle Accelerator > GPU T4 x2"
GPU_NAME = torch.cuda.get_device_name(0)
print(GPU_NAME, "| run dir", RUN_DIR)
sh(sys.executable, "-m", "pip", "install", "-q", "unsloth")
if IN_COLAB and COLAB_USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

## 3. Repo and prompts

In [ ]:
if not os.path.exists(REPO_DIR):
    sh("git", "clone", "-q", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "-q", "--all")
sh("git", "-C", REPO_DIR, "checkout", "-q", "--detach", f"origin/{REPO_REF}")
REPO_COMMIT = sh("git", "-C", REPO_DIR, "rev-parse", "HEAD")
sys.path.insert(0, os.path.join(REPO_DIR, "src"))
for m in [m for m in list(sys.modules) if m.startswith("finexpert")]:
    del sys.modules[m]
from finexpert.data.v4.generate import (generate_with_retries, judge_prompt, parse_judge, judge_accepts,
                                        write_calibration_sample)

def read_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]
ITEMS = {"explain": read_jsonl(f"{REPO_DIR}/data/v4/explain_prompts_train.jsonl"),
         "assess": read_jsonl(f"{REPO_DIR}/data/v4/assess_prompts_train.jsonl")}
if LIMIT:
    ITEMS = {k: v[:LIMIT] for k, v in ITEMS.items()}
print("repo", REPO_COMMIT[:7], {k: len(v) for k, v in ITEMS.items()})

## 4. Load the teacher

In [ ]:
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(model_name=TEACHER_MODEL, max_seq_length=4096,
                                                     dtype=None, load_in_4bit=True)
FastLanguageModel.for_inference(model)
tokenizer.padding_side = "left"
PAD_ID = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else tokenizer.eos_token_id
STOP_IDS = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids("<|im_end|>")]

## 5. Generate, check, retry
`generate()` is the only GPU-specific part; the checking and retry logic lives in `finexpert.data.v4.generate` and is
unit-tested in the repository.

In [ ]:
import re
from functools import partial

def chat(prompt):
    return ("<|im_start|>user\n" + prompt + "<|im_end|>\n<|im_start|>assistant\n"
            + ("<think>\n\n</think>\n\n" if NO_THINK else ""))

def generate(prompts, temperature, max_new_tokens):
    texts = [chat(p) for p in prompts]
    order = sorted(range(len(texts)), key=lambda i: len(texts[i]))
    out = [None] * len(texts)
    for b in range(0, len(order), GEN_BATCH_SIZE):
        idx = order[b:b + GEN_BATCH_SIZE]
        enc = tokenizer([texts[i] for i in idx], return_tensors="pt", padding=True).to("cuda")
        sampling = {"do_sample": True, "temperature": temperature, "top_p": 0.9} if temperature > 0 else {"do_sample": False}
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens, pad_token_id=PAD_ID, eos_token_id=STOP_IDS, **sampling)
        for i, g in zip(idx, gen):
            text = tokenizer.decode(g[enc["input_ids"].shape[1]:], skip_special_tokens=True)
            out[i] = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    return out

RESULTS, TIMING = {}, {}
for kind, items in ITEMS.items():
    t0 = time.time()
    RESULTS[kind] = generate_with_retries(kind, items, partial(generate, max_new_tokens=MAX_NEW_TOKENS[kind]))
    TIMING[kind] = round((time.time() - t0) / 60, 1)
    passed = sum(r["passed"] for r in RESULTS[kind])
    print(f"{kind}: {passed}/{len(items)} passed the automatic checks in {TIMING[kind]} min "
          f"(attempts: {dict(__import__('collections').Counter(r['attempts'] for r in RESULTS[kind]))})")

## 6. Judge the accepted items

In [ ]:
ROWS = []
for kind, items in ITEMS.items():
    by_id = {it["id"]: it for it in items}
    accepted = [r for r in RESULTS[kind] if r["passed"]]
    questions = [by_id[r["id"]]["question"] if kind == "explain" else by_id[r["id"]]["user"] for r in accepted]
    t0 = time.time()
    replies = generate([judge_prompt(kind, q, r["target"]) for q, r in zip(questions, accepted)], 0.0,
                       MAX_NEW_TOKENS["judge"])
    TIMING[f"judge_{kind}"] = round((time.time() - t0) / 60, 1)
    judged = {r["id"]: parse_judge(reply, kind) for r, reply in zip(accepted, replies)}
    for r in RESULTS[kind]:
        item = by_id[r["id"]]
        scores = judged.get(r["id"])
        ROWS.append({"id": r["id"], "kind": kind,
                     "question": item["question"] if kind == "explain" else item["user"],
                     "system": item["system"], "target": r["target"], "raw": r["raw"], "attempts": r["attempts"],
                     "checks_passed": r["passed"], "check_details": r["details"], "judge": scores,
                     "accepted": r["passed"] and judge_accepts(scores, kind),
                     **({"term": item["term"], "category": item["category"]} if kind == "explain" else
                        {"source_id": item["source_id"], "facts": item["facts"]})})
    n_ok = sum(1 for x in ROWS if x["kind"] == kind and x["accepted"])
    n_unparsed = sum(1 for v in judged.values() if v is None)
    print(f"{kind}: judge accepted {n_ok}/{len(accepted)} (unparseable judge replies: {n_unparsed})")

## 7. Save, calibration sheet, manifest

In [ ]:
import shutil, platform, datetime
from collections import Counter
for kind in ITEMS:
    with open(os.path.join(RUN_DIR, f"v4_{kind}_train.jsonl"), "w", encoding="utf-8") as f:
        for row in ROWS:
            if row["kind"] == kind:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
sample_ids = write_calibration_sample(ROWS, os.path.join(RUN_DIR, "calibration_sheet.csv"),
                                      os.path.join(RUN_DIR, "calibration_judge_key.json"))
stats = {kind: {"prompts": len(ITEMS[kind]),
                "passed_checks": sum(r["checks_passed"] for r in ROWS if r["kind"] == kind),
                "accepted": sum(r["accepted"] for r in ROWS if r["kind"] == kind),
                "attempts": dict(Counter(r["attempts"] for r in ROWS if r["kind"] == kind))} for kind in ITEMS}
manifest = {"run_name": RUN_NAME, "repo_commit": REPO_COMMIT, "teacher_model": TEACHER_MODEL, "no_think": NO_THINK,
            "limit": LIMIT, "stats": stats, "timing_minutes": TIMING, "calibration_sample": sample_ids, "gpu": GPU_NAME,
            "finished_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
            "python": platform.python_version()}
json.dump(manifest, open(os.path.join(RUN_DIR, "run_manifest.json"), "w"), indent=2)
archive = shutil.make_archive(os.path.join(WORK_DIR, RUN_NAME + "-results"), "zip", RUN_DIR)
print(json.dumps(stats, indent=2)); print("Results:", archive)
if IN_COLAB and COLAB_USE_DRIVE:
    shutil.copytree(RUN_DIR, f"/content/drive/MyDrive/finexpert/runs/{RUN_NAME}", dirs_exist_ok=True)
    print("Copied to Drive: MyDrive/finexpert/runs/" + RUN_NAME)